# Documentation Asteria Results

This notebook creates the numbers and plots used for the result documentation of the Asteria rocket simulation. Make sure all evaluated results are stored as pickle files in `evaluation_data/rocketpy/`.

In [ ]:
from pathlib import Path

from evaluation.core import EvaluationResult
from evaluation.visualizer import EvaluationVisualizer

EVAL_DATA_PATH = Path("../../evaluation_data/rocketpy")
NO_BIAS_EVAL_DATA_PATH = Path("../../evaluation_data/rocketpy/no_bias")
REFERENCE_SCENARIO = "asteria_evaluation"


def _load_dir(path: Path) -> list[EvaluationResult]:
    files = sorted(path.glob("*.pkl"))
    if not files:
        print(f"  {path.name}/: (empty — run 99_asteria_multi_evaluation first)")
        return []
    results = [EvaluationResult.load(p) for p in files]
    for result, p in zip(results, files):
        # workaround to avoid rerunning simulations...
        if result.metadata.get("simulation_metadata", {}).get("name") == "name":
            result.metadata["simulation_metadata"]["name"] = p.stem
    print(f"  {path.name}/: {[p.stem for p in files]}")
    return results


def _ref_eval(evals: list[EvaluationResult]) -> EvaluationResult | None:
    return next(
        (e for e in evals if e.metadata["simulation_metadata"]["name"] == REFERENCE_SCENARIO),
        evals[0] if evals else None,
    )


print("Biased variant evaluations:")
manual_evaluations: list[EvaluationResult] = _load_dir(EVAL_DATA_PATH / "manual")
sw_evaluations: list[EvaluationResult] = _load_dir(EVAL_DATA_PATH / "sw")
ema_evaluations: list[EvaluationResult] = _load_dir(EVAL_DATA_PATH / "ema")

manual_ref_evaluation: EvaluationResult = _ref_eval(manual_evaluations)
sw_ref_evaluation: EvaluationResult = _ref_eval(sw_evaluations)
ema_ref_evaluation: EvaluationResult = _ref_eval(ema_evaluations)

print("\nNo-bias variant evaluations:")
no_bias_manual_evaluations: list[EvaluationResult] = _load_dir(NO_BIAS_EVAL_DATA_PATH / "manual")
no_bias_sw_evaluations: list[EvaluationResult] = _load_dir(NO_BIAS_EVAL_DATA_PATH / "sw")
no_bias_ema_evaluations: list[EvaluationResult] = _load_dir(NO_BIAS_EVAL_DATA_PATH / "ema")

no_bias_manual_ref_evaluation: EvaluationResult = _ref_eval(no_bias_manual_evaluations)
no_bias_sw_ref_evaluation: EvaluationResult = _ref_eval(no_bias_sw_evaluations)
no_bias_ema_ref_evaluation: EvaluationResult = _ref_eval(no_bias_ema_evaluations)

## Flight Trajectories

In [ ]:
PLOT_VARIANT = "ema"  # "manual" | "sw" | "ema"
SHOW_ESTIMATE = True

_variant_map = {
    "manual": manual_evaluations,
    "sw": sw_evaluations,
    "ema": ema_evaluations,
}
_plot_results = _variant_map[PLOT_VARIANT]
_plot_labels = [e.metadata["simulation_metadata"]["name"] for e in _plot_results]

# fig = EvaluationVisualizer.plot_multiple_flight_trajectories(
#     _plot_results,
#     labels=_plot_labels,
#     show_estimate=SHOW_ESTIMATE,
#     title=f"Simulated ASTERIA Trajectories",
# )
# fig.update_layout(
#     scene=dict(
#         xaxis=dict(title_font=dict(size=14), tickfont=dict(size=14)),
#         yaxis=dict(title_font=dict(size=14), tickfont=dict(size=14)),
#         zaxis=dict(title_font=dict(size=14), tickfont=dict(size=14)),
#     ),
#     legend=dict(font=dict(size=16)),  # legend stays smaller
#     width=1200,
#     height=800,
# )
# fig.show()

## Accuracy

In [ ]:
import numpy as np
import pandas as pd

METRICS = ["position_rmse", "velocity_rmse", "attitude_rmse"]

# Use the manual (hand-tuned) variant as the primary accuracy reference
evaluations = manual_evaluations

scenarios = [ev.metadata["simulation_metadata"]["name"] for ev in evaluations]
print(f"Averaging over {len(evaluations)} scenarios: {scenarios}\n")

collected = {m: {"x": [], "y": [], "z": [], "3D": []} for m in METRICS}
for ev in evaluations:
    for m in METRICS:
        r = ev.metrics[m]
        collected[m]["3D"].append(r.value)
        for ax in ("x", "y", "z"):
            collected[m][ax].append(r.per_axis[ax])

rows = []
for m in METRICS:
    unit = evaluations[0].metrics[m].unit
    row = {"Metric": m, "Unit": unit}
    for col in ("x", "y", "z", "3D"):
        vals = np.array(collected[m][col])
        row[col] = f"{vals.mean():.4f} ± {vals.std():.4f}"
    rows.append(row)

df = pd.DataFrame(rows).set_index("Metric")
df

In [ ]:
visualizer = EvaluationVisualizer(manual_ref_evaluation)
ax = visualizer.plot_trajectory()
ax.view_init(azim=45)
# ax.figure.savefig("trajectory.png", dpi=300, bbox_inches="tight", pad_inches=0.3)

In [ ]:
fig = visualizer.plot_state_errors()
# fig.savefig("asteria_state_errors.png", dpi=600, bbox_inches="tight", pad_inches=0.3)

## Consistency

In [ ]:
visualizer = EvaluationVisualizer(manual_ref_evaluation)
ax = visualizer.plot_nees_groups_over_time()
# ax.get_figure().savefig("asteria_grouped_nees.png", dpi=300, bbox_inches="tight")

## Flight Phase Analysis

Breaks the full flight into time windows and re-uses the existing state-error plots with `xlim`.
Change `PHASE_RESULT` to compare a different variant.

In [ ]:
import numpy as np
import pandas as pd
from scipy.spatial.transform import Rotation

BOOST_DURATION = 9.536989  # seconds


def _phase_rmse(result: EvaluationResult, t_start: float, t_end: float) -> dict:
    t = result.estimate.time
    mask = (t >= t_start) & (t <= t_end)
    if not mask.any():
        return {"position [m]": np.nan, "velocity [m/s]": np.nan, "attitude [deg]": np.nan}

    pos_err = result.estimate.position[mask] - result.ground_truth.position[mask]
    vel_err = result.estimate.velocity[mask] - result.ground_truth.velocity[mask]
    r_gt = Rotation.from_quat(result.ground_truth.attitude[mask], scalar_first=True)
    r_est = Rotation.from_quat(result.estimate.attitude[mask], scalar_first=True)
    att_err = (r_gt * r_est.inv()).as_rotvec()

    return {
        "position [m]": np.sqrt(np.mean(np.sum(pos_err ** 2, axis=1))),
        "velocity [m/s]": np.sqrt(np.mean(np.sum(vel_err ** 2, axis=1))),
        "attitude [deg]": np.degrees(np.sqrt(np.mean(np.sum(att_err ** 2, axis=1)))),
    }


PHASE_METRICS = ["position [m]", "velocity [m/s]", "attitude [deg]"]
PHASES = ["Boost", "Coast", "Drogue", "Main Chute"]
phase_data = {phase: {m: [] for m in PHASE_METRICS} for phase in PHASES}

for ev in manual_evaluations:
    sim_meta = ev.metadata["simulation_metadata"]
    if sim_meta.get("drogue_inflation_time") is None or sim_meta.get("main_parachute_inflation_time") is None:
        continue

    t0 = float(ev.estimate.time[0])
    t_drogue = float(sim_meta["drogue_inflation_time"])
    t_main = float(sim_meta["main_parachute_inflation_time"])
    t_end = float(ev.estimate.time[-1])

    bounds = {
        "Boost": (t0, t0 + BOOST_DURATION),
        "Coast": (t0 + BOOST_DURATION, t_drogue),
        "Drogue": (t_drogue, t_main),
        "Main Chute": (t_main, t_end),
    }
    for phase, (ts, te) in bounds.items():
        rmse = _phase_rmse(ev, ts, te)
        for metric, val in rmse.items():
            phase_data[phase][metric].append(val)

rows = []
for phase in PHASES:
    row = {"Phase": phase}
    for metric in PHASE_METRICS:
        vals = np.array(phase_data[phase][metric])
        row[metric] = f"{np.nanmean(vals):.4f} ± {np.nanstd(vals):.4f}"
    rows.append(row)

df = pd.DataFrame(rows).set_index("Phase")
df

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

PHASE_RESULT = manual_ref_evaluation

sim_meta = PHASE_RESULT.metadata["simulation_metadata"]
t_launch = float(PHASE_RESULT.estimate.time[0])
t_drogue = float(sim_meta["drogue_inflation_time"])
t_main = float(sim_meta["main_parachute_inflation_time"])
t_touchdown = float(PHASE_RESULT.estimate.time[-1])

phases = {
    "Ascent": (t_launch, t_drogue),
    "Drogue phase": (t_drogue, t_main),
    "Main chute phase": (t_main, t_touchdown),
    "Descent": (t_drogue, t_touchdown),
}

print(f"Launch:     {t_launch:.1f} s")
print(f"Drogue:     {t_drogue:.1f} s  ({t_drogue - t_launch:.1f} s ascent)")
print(f"Main chute: {t_main:.1f} s  ({t_main - t_drogue:.1f} s drogue phase)")
print(f"Touchdown:  {t_touchdown:.1f} s  ({t_touchdown - t_main:.1f} s main chute phase)")


def _fit_ylim(ax: plt.Axes, t_start: float, t_end: float, margin: float = 0.05) -> None:
    y_min, y_max = np.inf, -np.inf
    for line in ax.lines:
        xd, yd = np.asarray(line.get_xdata()), np.asarray(line.get_ydata())
        mask = (xd >= t_start) & (xd <= t_end)
        if mask.any():
            y_min = min(y_min, np.nanmin(yd[mask]))
            y_max = max(y_max, np.nanmax(yd[mask]))
    if np.isfinite(y_min) and np.isfinite(y_max):
        pad = (y_max - y_min) * margin or 0.1
        ax.set_ylim(y_min - pad, y_max + pad)


def plot_phase(result: EvaluationResult, t_start: float, t_end: float, title: str = "") -> plt.Figure:
    fig = EvaluationVisualizer(result).plot_state_errors(show_events=False)
    for ax in fig.axes:
        ax.set_xlim(t_start, t_end)
        _fit_ylim(ax, t_start, t_end)
    if title:
        fig.suptitle(title, fontsize=12)
    fig.subplots_adjust(top=0.93)
    return fig

In [ ]:
fig = plot_phase(PHASE_RESULT, *phases["Ascent"], title="Ascent Phase")
# fig.savefig("asteria_ascent_error.png", dpi=600, bbox_inches="tight", pad_inches=0.3)

In [ ]:
fig = plot_phase(PHASE_RESULT, *phases["Descent"], title="Descent (drogue to touchdown)")
# fig.savefig("asteria_descent_error.png", dpi=600, bbox_inches="tight", pad_inches=0.3)

## Manual Optimization vs. Adaptive Measurement Noise

In [ ]:
import numpy as np
import pandas as pd

METRICS = ["position_rmse", "velocity_rmse", "attitude_rmse"]
VARIANT_EVALS = {
    "manual": manual_evaluations,
    "sw": sw_evaluations,
    "ema": ema_evaluations,
}

rows = []
for m in METRICS:
    row = {"Metric": m}
    for variant_name, evals in VARIANT_EVALS.items():
        vals = np.array([e.metrics[m].value for e in evals])
        row[variant_name] = f"{vals.mean():.4f} ± {vals.std():.4f}"
    row["Unit"] = evals[0].metrics[m].unit
    rows.append(row)

df = pd.DataFrame(rows).set_index("Metric")
df

In [ ]:
fig = EvaluationVisualizer.plot_state_error_norms_comparison(
    [manual_ref_evaluation, sw_ref_evaluation, ema_ref_evaluation],
    labels=["Static", "SW", "EMA"],
)
fig.savefig("asteria_adaptive_position_error.png", dpi=600, bbox_inches="tight", pad_inches=0.3)

In [ ]:
visualizer = EvaluationVisualizer(sw_ref_evaluation)
visualizer.plot_measurement_variances()
print("plots")

In [ ]:
EvaluationVisualizer.plot_state_error_norms_comparison(
    [manual_ref_evaluation, no_bias_sw_ref_evaluation, sw_ref_evaluation,
     no_bias_ema_ref_evaluation, ema_ref_evaluation],
    labels=["manual (biased)", "SW (no bias)", "SW", "EMA (no bias)", "EMA"],
)
print("combined plots: ")

In [ ]:
visualizer = EvaluationVisualizer(no_bias_sw_ref_evaluation)
visualizer.plot_measurement_variances()
print("plots")